In [ ]:
import cv2
from ultralytics import YOLO
import math
import numpy as np
from datetime import datetime

# Para Jupyter/Colab (opcional)
try:
    import IPython.display as ipd
    from IPython.display import display, Image as IPImage
    JUPYTER = True
except:
    JUPYTER = False

In [ ]:
import cv2
from ultralytics import YOLO
import math
import numpy as np

# ── Función para calcular ángulo ──────────────────────────────────────────────
def calcular_angulo(A, B, C):
    """
    Calcula el ángulo en el punto B formado por los puntos A-B-C
    """
    radianes = math.atan2(C[1] - B[1], C[0] - B[0]) - \
               math.atan2(A[1] - B[1], A[0] - B[0])
    angulo = abs(radianes * 180.0 / math.pi)
    if angulo > 180.0:
        angulo = 360 - angulo
    return angulo


# ── Función para verificar alineación corporal ────────────────────────────────
def verificar_alineacion(hombro, cadera, tobillo):
    """
    Verifica que el cuerpo esté alineado (espalda recta)
    Retorna True si la alineación es correcta
    """
    # Calcular el ángulo de la espalda
    angulo_espalda = calcular_angulo(tobillo, cadera, hombro)
    
    # Una buena flexión tiene la espalda casi recta (160-180 grados)
    return 160 <= angulo_espalda <= 190


# ── Función para verificar posición de caderas ────────────────────────────────
def verificar_caderas(cadera_y, hombro_y):
    """
    Verifica que las caderas no estén muy arriba o muy abajo
    """
    diferencia = abs(cadera_y - hombro_y)
    # Las caderas deben estar relativamente al mismo nivel que los hombros
    return diferencia < 150  # pixels


# ── Función principal de detección ────────────────────────────────────────────
def detector_flexiones_webcam():
    """
    Sistema completo de detección de flexiones con feedback en tiempo real
    """
    
    # ── Configuración ─────────────────────────────────────────────────────────
    model = YOLO("yolo11n-pose.pt")
    
    # Cambiar a webcam (0 = cámara por defecto, 1 = segunda cámara, etc.)
    cap = cv2.VideoCapture(0)
    
    # Configurar resolución (opcional)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    # Variables de estado
    contador_correctas = 0
    contador_incorrectas = 0
    estado = None
    estado_ant = None
    
    # Variables para feedback
    errores = []
    flexion_correcta = True
    
    print("🎥 Webcam iniciada. Presiona 'q' para salir.")
    print("📌 Colócate de perfil a la cámara para mejor detección.\n")
    
    # ── Bucle principal ───────────────────────────────────────────────────────
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("❌ Error al leer la webcam")
            break
        
        # Voltear la imagen horizontalmente (efecto espejo)
        frame = cv2.flip(frame, 1)
        
        # Detectar pose
        results = model(frame, verbose=False)
        
        # Reiniciar variables de feedback
        errores = []
        flexion_correcta = True
        
        for r in results:
            if r.keypoints is None or r.keypoints.xy.shape[0] == 0:
                continue
            
            kpts = r.keypoints.xy[0]
            conf = r.keypoints.conf[0]
            
            # Índices de keypoints
            IDX_HOMBRO_DER = 6
            IDX_CODO_DER = 8
            IDX_MUNECA_DER = 10
            IDX_CADERA_DER = 12
            IDX_RODILLA_DER = 14
            IDX_TOBILLO_DER = 16
            
            # Verificar que todos los puntos sean visibles
            puntos_necesarios = [IDX_HOMBRO_DER, IDX_CODO_DER, IDX_MUNECA_DER, 
                                IDX_CADERA_DER, IDX_RODILLA_DER, IDX_TOBILLO_DER]
            
            if all(conf[idx] > 0.5 for idx in puntos_necesarios):
                
                # Extraer coordenadas
                hombro = (int(kpts[IDX_HOMBRO_DER][0]), int(kpts[IDX_HOMBRO_DER][1]))
                codo = (int(kpts[IDX_CODO_DER][0]), int(kpts[IDX_CODO_DER][1]))
                muneca = (int(kpts[IDX_MUNECA_DER][0]), int(kpts[IDX_MUNECA_DER][1]))
                cadera = (int(kpts[IDX_CADERA_DER][0]), int(kpts[IDX_CADERA_DER][1]))
                rodilla = (int(kpts[IDX_RODILLA_DER][0]), int(kpts[IDX_RODILLA_DER][1]))
                tobillo = (int(kpts[IDX_TOBILLO_DER][0]), int(kpts[IDX_TOBILLO_DER][1]))
                
                # ── ANÁLISIS DE LA FLEXIÓN ────────────────────────────────────
                
                # 1. Ángulo del codo
                angulo_codo = calcular_angulo(hombro, codo, muneca)
                
                # 2. Verificar alineación corporal (espalda recta)
                alineacion_correcta = verificar_alineacion(hombro, cadera, tobillo)
                
                # 3. Verificar posición de caderas
                caderas_correctas = verificar_caderas(cadera[1], hombro[1])
                
                # 4. Verificar que las rodillas estén extendidas
                angulo_rodilla = calcular_angulo(cadera, rodilla, tobillo)
                rodillas_extendidas = angulo_rodilla > 160
                
                # ── DETERMINAR ESTADO ─────────────────────────────────────────
                
                if angulo_codo > 160:
                    estado = "arriba"
                elif angulo_codo < 90:
                    estado = "abajo"
                
                # ── VALIDAR FORMA CORRECTA ────────────────────────────────────
                
                flexion_correcta = True
                
                if not alineacion_correcta:
                    errores.append("¡Espalda recta!")
                    flexion_correcta = False
                
                if not caderas_correctas:
                    errores.append("¡Caderas alineadas!")
                    flexion_correcta = False
                
                if not rodillas_extendidas:
                    errores.append("¡Extiende las piernas!")
                    flexion_correcta = False
                
                # ── CONTAR REPETICIONES ───────────────────────────────────────
                
                if estado_ant == "abajo" and estado == "arriba":
                    if flexion_correcta:
                        contador_correctas += 1
                        print(f"✅ Flexión CORRECTA #{contador_correctas}")
                    else:
                        contador_incorrectas += 1
                        print(f"❌ Flexión INCORRECTA (Errores: {', '.join(errores)})")
                
                estado_ant = estado
                
                # ── COLORES SEGÚN ESTADO ──────────────────────────────────────
                
                # Verde si está correcta, Rojo si hay errores
                if flexion_correcta:
                    color_principal = (0, 255, 0)  # Verde
                    color_texto = (0, 255, 0)
                else:
                    color_principal = (0, 0, 255)  # Rojo
                    color_texto = (0, 0, 255)
                
                # Color según ángulo del codo
                if angulo_codo > 160:
                    color_codo = (0, 255, 0)  # Verde (arriba)
                elif angulo_codo < 90:
                    color_codo = (255, 0, 0)  # Azul (abajo)
                else:
                    color_codo = (0, 255, 255)  # Amarillo (medio)
                
                # ── DIBUJAR SKELETON ──────────────────────────────────────────
                
                # Brazo (color según forma)
                cv2.line(frame, hombro, codo, color_principal, 4)
                cv2.line(frame, codo, muneca, color_principal, 4)
                
                # Torso (color según alineación)
                color_torso = (0, 255, 0) if alineacion_correcta else (0, 0, 255)
                cv2.line(frame, hombro, cadera, color_torso, 4)
                
                # Piernas (color según extensión)
                color_piernas = (0, 255, 0) if rodillas_extendidas else (0, 0, 255)
                cv2.line(frame, cadera, rodilla, color_piernas, 4)
                cv2.line(frame, rodilla, tobillo, color_piernas, 4)
                
                # Círculos en articulaciones
                cv2.circle(frame, hombro, 8, (255, 0, 0), -1)
                cv2.circle(frame, codo, 8, color_codo, -1)
                cv2.circle(frame, muneca, 8, (255, 0, 0), -1)
                cv2.circle(frame, cadera, 8, (255, 255, 0), -1)
                cv2.circle(frame, rodilla, 8, (255, 255, 0), -1)
                cv2.circle(frame, tobillo, 8, (255, 255, 0), -1)
                
                # ── TEXTO INFORMATIVO ─────────────────────────────────────────
                
                # Ángulo del codo
                cv2.putText(frame, f"{int(angulo_codo)}°",
                           (codo[0] - 40, codo[1] - 15),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_codo, 2)
                
                # Estado
                cv2.putText(frame, f"Estado: {estado if estado else 'Posicionate'}",
                           (20, 50),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.0, color_texto, 2)
        
        # ── PANEL DE INFORMACIÓN ──────────────────────────────────────────────
        
        # Fondo semi-transparente para el panel
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 80), (500, 300), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
        
        # Contadores
        cv2.putText(frame, f"Correctas: {contador_correctas}",
                   (20, 120),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        
        cv2.putText(frame, f"Incorrectas: {contador_incorrectas}",
                   (20, 170),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        
        # Total
        total = contador_correctas + contador_incorrectas
        cv2.putText(frame, f"Total: {total}",
                   (20, 220),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        
        # Mostrar errores si hay
        if errores:
            y_pos = 270
            for error in errores:
                cv2.putText(frame, f"⚠ {error}",
                           (20, y_pos),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                y_pos += 35
        else:
            cv2.putText(frame, "✓ Forma correcta!",
                       (20, 270),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        # Instrucciones
        cv2.putText(frame, "Presiona 'q' para salir | 'r' para resetear",
                   (20, frame.shape[0] - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # ── MOSTRAR FRAME ─────────────────────────────────────────────────────
        
        cv2.imshow('Detector de Flexiones - Tiempo Real', frame)
        
        # ── CONTROLES DE TECLADO ──────────────────────────────────────────────
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('q'):
            print("\n🛑 Deteniendo detector...")
            break
        elif key == ord('r'):
            contador_correctas = 0
            contador_incorrectas = 0
            print("\n🔄 Contadores reseteados")
    
    # ── FINALIZAR ─────────────────────────────────────────────────────────────
    
    cap.release()
    cv2.destroyAllWindows()
    
    print(f"\n📊 RESUMEN FINAL:")
    print(f"   ✅ Flexiones correctas: {contador_correctas}")
    print(f"   ❌ Flexiones incorrectas: {contador_incorrectas}")
    print(f"   📈 Total: {contador_correctas + contador_incorrectas}")
    
    if contador_correctas + contador_incorrectas > 0:
        porcentaje = (contador_correctas / (contador_correctas + contador_incorrectas)) * 100
        print(f"   🎯 Porcentaje de precisión: {porcentaje:.1f}%")


# ── EJECUTAR ──────────────────────────────────────────────────────────────────
detector_flexiones_webcam()